# Data Loading and Preprocessing

### Load Libraries

In [1]:
import pandas as pd
import sqlite3
import joblib
import os
from tqdm.notebook import tqdm
import os
import numpy as np

# set working directory
os.chdir("/home/jovyan/dsp/")

## Load Data by Year

In [2]:
data = joblib.load('data/embeddings/ipc_embeddings_2006.pkl')

In [ ]:
patents_all = list(data.keys())

In [ ]:
# remove _claims and _abstract to get patent ids
patent_ids = list(set([int(patent.split('_')[0]) for patent in patents_all]))
# remove duplicates
patent_ids = list(set(patent_ids))
print(f"Number of unique patents: {len(patent_ids)}")

Number of unique patents: 1102633


In [ ]:
patents_by_year = {}
conn = sqlite3.connect('data/patent.db')
cur = conn.cursor()
for year in tqdm(range(1980, 2024)):
    query = f"""
    SELECT id FROM patent p
    WHERE strftime('%Y', pub_date) = '{year}'
    ;
    """
    df_year = pd.read_sql_query(query, conn)
    patents_by_year[year] = df_year['id'].tolist()
conn.close()

  0%|          | 0/44 [00:00<?, ?it/s]

100%|██████████| 44/44 [00:45<00:00,  1.04s/it]


In [ ]:
type(patent_ids[0])

str

In [ ]:
# check which patents are also in the embedding data
patents_in_embeddings_by_year = {}
for year in range(1980, 2024):
    patents_in_embeddings = list(set(patents_by_year[year]).intersection(set(patent_ids)))
    patents_in_embeddings_by_year[year] = patents_in_embeddings
    print(f"Year {year}: {len(patents_in_embeddings)} patents in embeddings out of {len(patents_by_year[year])} total patents.")

Year 1980: 0 patents in embeddings out of 6752 total patents.
Year 1981: 0 patents in embeddings out of 12213 total patents.
Year 1982: 0 patents in embeddings out of 15706 total patents.
Year 1983: 0 patents in embeddings out of 18181 total patents.
Year 1984: 0 patents in embeddings out of 22718 total patents.
Year 1985: 0 patents in embeddings out of 27532 total patents.
Year 1986: 0 patents in embeddings out of 31885 total patents.
Year 1987: 0 patents in embeddings out of 35369 total patents.
Year 1988: 0 patents in embeddings out of 39815 total patents.
Year 1989: 0 patents in embeddings out of 47044 total patents.
Year 1990: 0 patents in embeddings out of 53177 total patents.
Year 1991: 0 patents in embeddings out of 54063 total patents.
Year 1992: 0 patents in embeddings out of 50026 total patents.
Year 1993: 0 patents in embeddings out of 41500 total patents.
Year 1994: 0 patents in embeddings out of 35181 total patents.
Year 1995: 0 patents in embeddings out of 35269 total pa

In [ ]:
# create copy of data file so we can remove patents year by year
data_copy = data.copy()

# create embedding files for patents in each year
for year in tqdm(range(1980, 2024), desc="Year: ", position=0):
    patents_in_embeddings = patents_in_embeddings_by_year[year]
    data_year = {}
    for patent_id in tqdm(patents_in_embeddings, leave=False, desc = "Patents: ", position=1):
        # add abstract and claims embeddings to year data if they exist
        if f"{patent_id}_abstract" in data_copy:
            data_year[f"{patent_id}_abstract"] = data_copy[f"{patent_id}_abstract"]
            del data_copy[f"{patent_id}_abstract"]
        if f"{patent_id}_claims" in data_copy:
            data_year[f"{patent_id}_claims"] = data_copy[f"{patent_id}_claims"]
            del data_copy[f"{patent_id}_claims"]
    joblib.dump(data_year, f"data/embeddings/by_year/embeddings_{year}.pkl")

Year: 100%|██████████| 44/44 [03:37<00:00,  4.94s/it] 


## Calculate IPC Embeddings by Year

In [3]:
def load_embedding_data(year):
    data_year = joblib.load(f"data/embeddings/by_year/embeddings_{year}.pkl")
    return data_year

def calc_mean_embedding(embeddings):
    if len(embeddings) == 0:
        return None
    return np.mean(np.array(embeddings), axis=0)

def get_ipc_to_patent_mapping(data):
    ipc_to_patent = {}
    for patent_key in data.keys():
        ipc_codes = list(data[patent_key].keys())
        for ipc in ipc_codes:
            if ipc not in ipc_to_patent:
                ipc_to_patent[ipc] = []
            ipc_to_patent[ipc].append(patent_key)
    return ipc_to_patent

In [10]:
mapping

{'a01g7': ['1704769_abstract',
  '1704769_claims',
  '1723844_abstract',
  '1723844_claims',
  '1736053_abstract',
  '1736053_claims',
  '1611788_abstract',
  '1611788_claims',
  '1618781_abstract',
  '1618781_claims',
  '1621069_abstract',
  '1621069_claims',
  '1639884_abstract',
  '1639884_claims',
  '1679367_abstract',
  '1679367_claims',
  '1700520_abstract',
  '1700520_claims',
  '1700919_abstract',
  '1700919_claims'],
 'a01g13': ['1704769_abstract',
  '1704769_claims',
  '1716747_abstract',
  '1716747_claims',
  '1621071_abstract',
  '1621071_claims',
  '1625785_abstract',
  '1625785_claims',
  '1634492_abstract',
  '1634492_claims',
  '1647342_abstract',
  '1647342_claims',
  '1654925_abstract',
  '1654925_claims',
  '1678999_abstract',
  '1678999_claims'],
 'a01g1': ['1704769_abstract',
  '1704769_claims',
  '1721510_abstract',
  '1721510_claims',
  '1728416_abstract',
  '1728416_claims',
  '1736053_abstract',
  '1736053_claims',
  '1617008_abstract',
  '1617008_claims',
  '1

In [11]:
for year in tqdm(range(1980, 2024), desc="Years"):
    data_year = load_embedding_data(year)
    ipc_to_patent = get_ipc_to_patent_mapping(data_year)
    if ipc_to_patent is None:
        tqdm.write(f"No IPC mapping for year {year}")
        continue

    ipc_mean_embeddings = {}
    
    

    for ipc in ipc_to_patent.keys():
        embeddings = []
        
        doc_list = [doc for doc in ipc_to_patent[ipc] if '_abstract' in doc]
        for patent_key in doc_list:
            if ipc not in data_year[patent_key]:
                tqdm.write(f"Warning: IPC {ipc} not found in patent {patent_key} ({year})")
            else:
                embeddings.append(data_year[patent_key][ipc])

        if embeddings:
            ipc_mean_embeddings[ipc] = calc_mean_embedding(embeddings)

    joblib.dump(ipc_mean_embeddings, f"data/embeddings/ipc_mean_year_abstract/ipc_mean_{year}.pkl")

Years:   0%|          | 0/44 [00:00<?, ?it/s]